## Making border files for ROIs

In [1]:
import os, pickle, subprocess
import matplotlib.pyplot as plt
import matplotlib as mpl
from py_util_dx.py_utils import setProjectPath
from py_util_dx.data_utils import get_roi_vtx_from_fs32k, get_roi_pacels, get_glasser_labels
from scipy.stats import ttest_1samp
import Functional_Fusion.atlas_map as am 
import seaborn as sns 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import matplotlib as mpl 
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
import nibabel as nib
from visualizations import plot_flatmap_labels
from nitools.cifti import surf_from_cifti, split_cifti_to_giftis
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import MDS
import matplotlib.colors as mcolors
from scipy.linalg import orthogonal_procrustes

In [2]:
mpl.rcParams['font.family'] = 'helvetica'   # Global font family
mpl.rcParams['font.size'] = 12              # Global font size 
full_width = 8.5
half_width = full_width/2

In [3]:
projectPath, mainResultsPath = setProjectPath()
dataset_name = 'MDTB' # or Demand
surface_helpers_dir = os.path.join(projectPath, 'surface_helpers')
resultsPath = os.path.join(mainResultsPath, '10_plot_ROI_borders')
if not os.path.exists(resultsPath):
    os.makedirs(resultsPath) 

In [4]:
atlas_str = 'fs32k'
atlas, ainf = am.get_atlas(atlas_str)
atlas_name = 'glasser'

flat_surf_L = os.path.join(surface_helpers_dir, 'fs_LR.32k.L.flat.surf.gii')
flat_surf_R = os.path.join(surface_helpers_dir, 'fs_LR.32k.R.flat.surf.gii')

In [5]:
# wb_command = f'wb_command -surface-merge fs_LR.32k.LR.flat.surf.gii -surface {flat_surf_L} -surface {flat_surf_R}'
# subprocess.run(wb_command, shell=True) 

In [6]:
atlas_fname = [os.path.join(surface_helpers_dir, f'{atlas_name}.L.label.gii'), os.path.join(surface_helpers_dir, f'{atlas_name}.R.label.gii')]
U = atlas.read_data(atlas_fname)

In [7]:
U = np.zeros_like(U) 
U 

array([0, 0, 0, ..., 0, 0, 0], shape=(59518,), dtype=int32)

In [8]:
included_vtx_inds_LR, included_vtx_inds_L, included_vtx_inds_R, excluded_vtx_inds_LR = get_roi_vtx_from_fs32k('PFC') 

In [9]:
U[included_vtx_inds_LR] = 1

In [10]:
included_vtx_inds_LR, included_vtx_inds_L, included_vtx_inds_R, excluded_vtx_inds_LR = get_roi_vtx_from_fs32k('parietal') 
U[included_vtx_inds_LR] = 1

In [11]:
cifti = atlas.data_to_cifti(U.reshape(1, -1))
cifti 

In [12]:
metric_file = os.path.join(resultsPath, f'roi_border.dscalar.nii')
# metric_file = os.path.join(resultsPath, f'roi_border.dlabel.nii')
nib.save(cifti, metric_file)

In [13]:
label_file = os.path.join(resultsPath, 'roi_border.dlabel.nii')
text_file = os.path.join(resultsPath, f'roi_labels.txt')
wb_command = f'wb_command -cifti-label-import {metric_file} {text_file} {label_file}' 

subprocess.run(wb_command, shell=True) 

CompletedProcess(args='wb_command -cifti-label-import /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_border.dscalar.nii /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_labels.txt /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_border.dlabel.nii', returncode=0)

In [16]:
border_file = os.path.join(resultsPath, "roi_border.border")
wb_command = f'wb_command -cifti-label-to-border {label_file} -border {flat_surf_L} {border_file}'

subprocess.run(wb_command, shell=True) 

CompletedProcess(args='wb_command -cifti-label-to-border /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_border.dlabel.nii -border /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../surface_helpers/fs_LR.32k.L.flat.surf.gii /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_border.border', returncode=0)

In [18]:
border_file_v1 = os.path.join(resultsPath, f'roi_borders_v1.border')

wb_command = f'wb_command -file-convert -border-version-convert {border_file} 1 {border_file_v1}'
subprocess.run(wb_command, shell=True) 

CompletedProcess(args='wb_command -file-convert -border-version-convert /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_border.border 1 /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_borders_v1.border', returncode=0)

In [25]:
import xml.etree.ElementTree as ET


def convert_border_v3_to_compatible(input_file, output_file):
    """Convert border file from Version 3 to structure expected by read_borders()."""
    
    # Parse input file
    tree = ET.parse(input_file)
    root = tree.getroot()
    
    # Check version
    version = root.attrib.get('Version', '1')
    print(f"Input file version: {version}")
    
    # Create new root - keep Version 3 but restructure
    new_root = ET.Element('BorderFile')
    new_root.attrib['Version'] = '1'
    new_root.attrib['Structure'] = root.attrib.get('Structure', 'CORTEX_LEFT')
    new_root.attrib['SurfaceNumberOfVertices'] = root.attrib.get('SurfaceNumberOfVertices', '32492')
    
    # Process each Class
    for class_elem in root.findall('Class'):
        # Create new Class element
        new_class = ET.SubElement(new_root, 'Class')
        new_class.attrib['Name'] = class_elem.attrib.get('Name', 'row 000')
        new_class.attrib['Red'] = class_elem.attrib.get('Red', '0')
        new_class.attrib['Green'] = class_elem.attrib.get('Green', '0')
        new_class.attrib['Blue'] = class_elem.attrib.get('Blue', '0')
        
        # Process each Border
        for border in class_elem.findall('Border'):
            # Create new Border element
            new_border = ET.SubElement(new_class, 'Border')
            new_border.attrib['Name'] = border.attrib.get('Name', 'Border')
            new_border.attrib['Red'] = border.attrib.get('Red', '1')
            new_border.attrib['Green'] = border.attrib.get('Green', '0')
            new_border.attrib['Blue'] = border.attrib.get('Blue', '0')
            
            # Process each BorderPart
            for border_part in border.findall('BorderPart'):
                # This is the key: each BorderPart becomes a direct child element
                # that contains Vertices and Weights
                new_part = ET.SubElement(new_border, 'SurfaceProjectedItem')
                
                # Get vertices and weights from Version 3 format
                vertices_elem = border_part.find('Vertices')
                weights_elem = border_part.find('Weights')
                
                if vertices_elem is not None and vertices_elem.text:
                    # Create Vertices element
                    vert_elem = ET.SubElement(new_part, 'Vertices')
                    vert_elem.text = vertices_elem.text.strip()
                    
                    # Create Weights element
                    if weights_elem is not None and weights_elem.text:
                        weight_elem = ET.SubElement(new_part, 'Weights')
                        weight_elem.text = weights_elem.text.strip()
    
    # Write output file
    tree_out = ET.ElementTree(new_root)
    ET.indent(tree_out, space='    ')
    tree_out.write(output_file, encoding='UTF-8', xml_declaration=True)
    
    print(f"✓ Converted {input_file}")
    print(f"✓ Output saved to {output_file}")
    print(f"\nStructure:")
    print(f"  Classes: {len(new_root.findall('Class'))}")
    for cls in new_root.findall('Class'):
        print(f"    - {cls.attrib['Name']}: {len(cls.findall('Border'))} borders")
        for border in cls.findall('Border'):
            parts = len(border.findall('SurfaceProjectedItem'))
            print(f"      - {border.attrib['Name']}: {parts} parts")

In [26]:
convert_border_v3_to_compatible(border_file, border_file_v1) 

Input file version: 3
✓ Converted /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_border.border
✓ Output saved to /Users/jkderrick028/Documents/Projects/7TfMRI/pfc_parcellation/code/../results/10_plot_ROI_borders/roi_borders_v1.border

Structure:
  Classes: 1
    - row 000: 1 borders
      - ROI: 7 parts


In [ ]:
# metric_file_left = os.path.join(resultsPath, f'roi_border.left.shape.gii')
metric_file_left = os.path.join(resultsPath, f'roi_border.left.label.gii')
# metric_file_left = os.path.join(resultsPath, f'roi_border.left.dlabel.nii')
wb_command = f'wb_command -cifti-separate {metric_file} COLUMN -metric CORTEX_LEFT {metric_file_left}' 
subprocess.run(wb_command, shell=True) 

In [15]:
# gii_file = os.path.join(resultsPath, f'roi_border.surf.gii')
# wb_command = f'wb_command -cifti-convert -to-gifti-ext {metric_file} {gii_file}' 
# subprocess.run(wb_command, shell=True) 

In [ ]:
# border_file = os.path.join(resultsPath, "roi_border.border")
# # flat_surf_L = os.path.join(surface_helpers_dir, f'sub-01.L.sulc.32k_fs_LR.shape.gii')
# wb_command = f'wb_command -metric-rois-to-border {flat_surf_L} {metric_file_left} PFC_Parietal {border_file}'

# subprocess.run(wb_command, shell=True) 

In [ ]:
border_file = os.path.join(resultsPath, "roi_border.border")
# flat_surf_L = os.path.join(surface_helpers_dir, f'sub-01.L.sulc.32k_fs_LR.shape.gii')
wb_command = f'wb_command -cifti-label-to-border {metric_file_left} -border {flat_surf_L} {border_file} -legacy'

subprocess.run(wb_command, shell=True) 

In [ ]:
# metric_file_border = os.path.join(resultsPath, "roi_border.left.func.gii")
# wb_command = f'wb_command -label-to-metric {flat_surf_L} {metric_file_left} {metric_file_border} -name PFC_Parietal'

# subprocess.run(wb_command, shell=True) 

In [ ]:
border_file_txt = os.path.join(resultsPath, "roi_border.txt")
wb_command = f'wb_command -border-export-color-table {border_file} {border_file_txt}'

subprocess.run(wb_command, shell=True) 

In [ ]:
border_resampled = os.path.join(resultsPath, "roi_border_resampled.border")

wb_command = f'wb_command -border-resample {border_file} {flat_surf_L} {flat_surf_L} {border_resampled}' 

subprocess.run(wb_command, shell=True) 